In [14]:
# @title Domyślny tekst tytułu
import pandas as pd

# 1. Wczytanie danych
df = pd.read_csv("Online_Retail.csv", encoding="ISO-8859-1")

df.head()
df.info()

# 2. Czyszczenie danych

# usunięcie braków CustomerID
df = df.dropna(subset=["CustomerID"])

# usunięcie anulowanych transakcji
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# usunięcie błędnych wartości
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

# konwersja daty
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# usunięcie duplikatów
df = df.drop_duplicates()

# dodanie Revenue
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


/tmp/ipykernel_2038/686523650.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


Wybrano ziarno na poziomie pojedynczej pozycji faktury (InvoiceNo + StockCode).
Pozwala to analizować sprzedaż na poziomie konkretnych produktów, klientów oraz czasu.
Dzięki temu możliwe jest wykonywanie szczegółowych analiz, np. które produkty sprzedają się najlepiej w danym kraju.

Przykład analizy: identyfikacja najlepiej sprzedających się produktów w danym miesiącu.

Zaprojektowano schemat gwiazdy składający się z tabeli faktów FactSales oraz trzech wymiarów: DimCustomer, DimProduct oraz DimDate.

Tabela FactSales przechowuje miary sprzedaży, takie jak Quantity oraz Revenue, oraz klucze obce do tabel wymiarów.

DimCustomer zawiera informacje o klientach (CustomerID, Country),
DimProduct opisuje produkty (StockCode, Description),
a DimDate zawiera informacje o czasie (data, rok, miesiąc, dzień).

In [15]:
# DimCustomer
dim_customer = df[["CustomerID", "Country"]].drop_duplicates().reset_index(drop=True)
dim_customer["customer_key"] = dim_customer.index + 1

# DimProduct
dim_product = df[["StockCode", "Description"]].drop_duplicates().reset_index(drop=True)
dim_product["product_key"] = dim_product.index + 1

# DimDate
dim_date = df[["InvoiceDate"]].drop_duplicates().reset_index(drop=True)
dim_date["date_key"] = dim_date.index + 1
dim_date["Year"] = dim_date["InvoiceDate"].dt.year
dim_date["Month"] = dim_date["InvoiceDate"].dt.month
dim_date["Day"] = dim_date["InvoiceDate"].dt.day

In [19]:
# łączenie z wymiarami
fact = df.merge(dim_customer, on=["CustomerID", "Country"])
fact = fact.merge(dim_product, on=["StockCode", "Description"])
fact = fact.merge(dim_date, on=["InvoiceDate"])

# wybór kolumn do fact table
fact_sales = fact[[
    "customer_key",
    "product_key",
    "date_key",
    "Quantity",
    "Revenue"
]]

fact_sales.head()

,customer_key,product_key,date_key,Quantity,Revenue
0,1,1,1,6,15.30
1,1,2,1,6,20.34
2,1,3,1,8,22.00
3,1,4,1,6,20.34
4,1,5,1,6,20.34


Zastosowano SCD typu 1 dla wymiaru DimCustomer.
Oznacza to, że zmiany w danych klienta (np. zmiana kraju) nadpisują poprzednie wartości i nie jest przechowywana historia zmian.

Wybrano to rozwiązanie ze względu na prostotę implementacji oraz brak konieczności analiz historycznych zmian danych klientów.

Podczas realizacji zadania zaprojektowano hurtownię danych w schemacie gwiazdy oraz przygotowano dane wejściowe poprzez ich oczyszczenie i transformację.

Najważniejszą decyzją było określenie ziarna danych, które wybrano na poziomie pozycji faktury, co zapewnia dużą elastyczność analiz.

Największą trudnością było odpowiednie przygotowanie danych oraz zrozumienie relacji między tabelą faktów a wymiarami.

Dzięki temu zadaniu nauczono się projektowania schematu gwiazdy, pracy z Pandas oraz podstaw modelowania hurtowni danych.